In [1]:
# 1. Import libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

nltk.download("stopwords")

# 2. Load dataset
df = pd.read_csv("bans_basket.csv")
print(df.head())
print(df.columns)

# 3. Select text column
text_col = "text"   # CHANGE THIS if your column has another name
df = df.dropna(subset=[text_col])

# 4. NLP cleaning
stop = set(stopwords.words("english"))

def clean(x):
    x = re.sub(r"[^a-zA-Z\s]", "", str(x).lower())
    return " ".join(w for w in x.split() if w not in stop)

df["clean_text"] = df[text_col].apply(clean)

# 5. TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(df["clean_text"])

# 6. K-Means
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# 7. Show clusters
print(df[[text_col, "cluster"]].head(20))

# 8. Display important words in each cluster
words = tfidf.get_feature_names_out()

for i in range(k):
    top = model.cluster_centers_[i].argsort()[-10:][::-1]
    print("Cluster", i, ":", ", ".join(words[j] for j in top))

# 9. Visualisation
pca = PCA(n_components=2)
points = pca.fit_transform(X.toarray())

plt.scatter(points[:,0], points[:,1], c=df["cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means NLP Clustering")
plt.show()

# 10. Save result
df.to_csv("bans_basket_clustered.csv", index=False)

   Id Banned hero
0   1      Rumble
1   1    Kassadin
2   1   Lissandra
3   1    Tristana
4   1     Leblanc
Index(['Id', 'Banned hero'], dtype='str')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


KeyError: ['text']